# Spinal cord (SC) — Baysor outputs → AnnData → clustering

The two **non-stroke** regions in the Fan / CG MERSCOPE run are the spinal cord samples
`SC_uninjured_1` and `SC_uninjured_2`. This notebook handles them on their own:

1. Rebuild a `.h5ad` per SC region from its Baysor `segmentation.csv` / `segmentation_cell_stats.csv`
   (same logic as `baysor_stroke_to_h5ad.ipynb`, just restricted to the SC samples).
2. Concatenate the two into `sc_all.h5ad`.
3. Cluster (QC → normalize → PCA → neighbors → Leiden → UMAP → markers → spatial),
   same pipeline as `baysor_stroke_clustering.ipynb`. Save `sc_all_clustered.h5ad`.

**Kernel:** `sc` (conda env with scanpy / pandas / numpy).

> Re-run safe: nothing here writes back to the Baysor outputs; only new `sc_*.h5ad` files are produced.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

# Must match `output_root` + `param_tag` from baysor_stroke_batch.ipynb
seg_root  = Path("/Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation")
param_tag = "m50_s4"

# Where the .h5ad files are written
out_dir = Path("/Volumes/T7/Stroke_merscop_Fan_CG/h5ad")
out_dir.mkdir(parents=True, exist_ok=True)

# The two non-stroke (spinal cord) regions
SC_SAMPLES = ["SC_uninjured_1", "SC_uninjured_2"]

assert seg_root.exists(), f"Segmentation root not found: {seg_root}"

## 1. Rebuild a `.h5ad` for each SC region

Baysor's `segmentation.csv` is one row per transcript with a `cell` column (0 / empty = background noise,
which we drop). We cross-tabulate transcripts into a cell × gene matrix and align it with the cell-stats table.

In [ ]:
def find_outputs(root: Path, tag: str, only=None):
    found = []
    for seg in sorted(root.glob(f"*/{tag}/segmentation_segmentation.csv")):
        if seg.name.startswith("._"):
            continue
        sample = seg.parent.parent.name           # <root>/<sample>/<tag>/segmentation.csv
        if only is not None and sample not in only:
            continue
        stats = seg.parent / "segmentation_cell_stats.csv"
        if seg.stat().st_size > 0 and stats.exists():
            found.append({"sample": sample, "seg": seg, "stats": stats})
    return found

outputs = find_outputs(seg_root, param_tag, only=SC_SAMPLES)
print(f"Found {len(outputs)} SC region(s):")
for o in outputs:
    print("  ", o["sample"])

missing = set(SC_SAMPLES) - {o["sample"] for o in outputs}
if missing:
    print("WARNING: no Baysor output found for:", sorted(missing))

In [ ]:
def build_adata(sample: str, seg: Path, stats: Path) -> ad.AnnData:
    tx = pd.read_csv(seg)
    tx = tx[tx["cell"].notna()]
    tx["cell"] = tx["cell"].astype(str)
    # Drop the background label (Baysor uses 0 / empty for noise transcripts)
    tx = tx[~tx["cell"].isin(["0", "", "nan"])]
    tx["gene"] = tx["gene"].astype(str)

    counts = pd.crosstab(tx["cell"], tx["gene"])

    st = pd.read_csv(stats)
    st["cell"] = st["cell"].astype(str)
    st = st.set_index("cell").reindex(counts.index)

    adata = ad.AnnData(
        X=counts.values.astype(np.float32),
        obs=st,
        var=pd.DataFrame(index=counts.columns),
    )
    adata.obs_names = [f"{sample}_{c}" for c in counts.index]
    adata.obs["sample"] = sample

    # Spatial centroid -> .obsm['spatial'] (Baysor stats name the centroid x/y)
    xcol = next((c for c in ("x", "centroid_x", "global_x") if c in adata.obs), None)
    ycol = next((c for c in ("y", "centroid_y", "global_y") if c in adata.obs), None)
    if xcol and ycol:
        adata.obsm["spatial"] = adata.obs[[xcol, ycol]].to_numpy(dtype=float)
    return adata


adatas = {}
for o in outputs:
    a = build_adata(o["sample"], o["seg"], o["stats"])
    adatas[o["sample"]] = a
    out_path = out_dir / f"{o['sample']}.h5ad"
    a.write_h5ad(out_path)
    print(f"{o['sample']:20s}  cells={a.n_obs:6d}  genes={a.n_vars:4d}  ->  {out_path.name}")

## 2. Concatenate the two SC regions into `sc_all.h5ad`

Outer join on genes so small panel differences still align; `sample` in `.obs` keeps the two regions separable.

In [ ]:
if adatas:
    combined = ad.concat(adatas, join="outer", label="sample", index_unique=None)
    combined.X = np.nan_to_num(combined.X)
    sc_all_path = out_dir / "sc_all.h5ad"
    combined.write_h5ad(sc_all_path)
    print(combined)
    print("\nWrote", sc_all_path)
else:
    print("No SC regions found — run baysor_stroke_batch.ipynb first.")

## 3. Clustering

Same pipeline as `baysor_stroke_clustering.ipynb`. This is a **targeted 500-gene panel**, so we cluster on
*all* genes (no HVG selection).

In [ ]:
adata = sc.read_h5ad(out_dir / "sc_all.h5ad")
print(adata)
print(adata.obs["sample"].value_counts())

### 3.0 Save the raw data (per sample)

Before any QC filtering or normalization, write each SC sample's **untouched integer counts** to its own
`raw/<sample>_raw.h5ad`. `X` at this point is still raw counts, so these files are a faithful snapshot of
every cell before we drop or rescale anything. The clustered object additionally keeps raw counts in
`layers['counts']`.

In [ ]:
raw_dir = out_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

samples_raw = adata.obs["sample"].cat.categories if hasattr(adata.obs["sample"], "cat") \
    else sorted(adata.obs["sample"].unique())

for s in samples_raw:
    sub = adata[adata.obs["sample"] == s].copy()   # untouched raw counts in .X
    p = raw_dir / f"{s}_raw.h5ad"
    sub.write_h5ad(p)
    print(f"{s:20s}  cells={sub.n_obs:6d}  genes={sub.n_vars:4d}  ->  {p}")

### 3.1 QC

Imaging-based panels are small, so cells carry few transcripts. Look at the distributions, then drop cells
that are too sparse to type reliably and genes seen in almost no cell.

In [ ]:
adata.var_names_make_unique()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].hist(adata.obs["total_counts"], bins=100); axs[0].set_xlabel("total_counts"); axs[0].set_yscale("log")
axs[1].hist(adata.obs["n_genes_by_counts"], bins=100); axs[1].set_xlabel("n_genes_by_counts")
if "area" in adata.obs:
    axs[2].hist(adata.obs["area"], bins=100); axs[2].set_xlabel("area"); axs[2].set_yscale("log")
plt.tight_layout(); plt.show()

adata.obs[["total_counts", "n_genes_by_counts"]].describe()

In [ ]:
# --- filtering thresholds (tune to the histograms above) ---
MIN_COUNTS = 10     # min transcripts per cell
MIN_GENES  = 2      # min distinct genes per cell
MIN_CELLS  = 10     # min cells expressing a gene to keep it

n0 = adata.n_obs
sc.pp.filter_cells(adata, min_counts=MIN_COUNTS)
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS)
print(f"cells: {n0} -> {adata.n_obs}  ({n0 - adata.n_obs} removed)")
print(f"genes kept: {adata.n_vars}")

### 3.2 Normalize

Keep raw counts in `layers['counts']`, then total-count normalize and log1p.

In [ ]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata)        # median-count normalization
sc.pp.log1p(adata)
adata.raw = adata                    # log-normalized snapshot for plotting / DE

# Targeted panel: use all genes.
sc.tl.pca(adata, n_comps=50)
sc.pl.pca_variance_ratio(adata, n_pcs=50)

### 3.3 (Optional) batch correction across the two regions

If clusters separate by `sample` rather than biology, run Harmony on the PCA embedding. Leave unrun to
cluster on the raw PCA. (`harmonypy` is pip-installed on first use.)

In [ ]:
USE_HARMONY = False   # set True to batch-correct on 'sample'

rep = "X_pca"
if USE_HARMONY:
    try:
        import harmonypy  # noqa: F401
    except ModuleNotFoundError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "harmonypy"])
    sc.external.pp.harmony_integrate(adata, key="sample")
    rep = "X_pca_harmony"
print("Using representation:", rep)

### 3.4 Neighbors → Leiden → UMAP

In [ ]:
RESOLUTION = 1.0    # higher -> more clusters

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, use_rep=rep)
sc.tl.leiden(adata, resolution=RESOLUTION, key_added="leiden")
sc.tl.umap(adata)
print("n clusters:", adata.obs["leiden"].nunique())
adata.obs["leiden"].value_counts().sort_index()

In [ ]:
sc.pl.umap(adata, color=["leiden", "sample"], wspace=0.35)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts"], wspace=0.35)

### 3.5 Marker genes per cluster

Wilcoxon rank-sum on the log-normalized data (`.raw`) to find each cluster's top genes — the starting point
for annotating cell types.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon", use_raw=True)
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False, fontsize=8)

top = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top

In [ ]:
# Dotplot of the top markers per cluster
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=4, use_raw=True)

### 3.6 Spatial map of clusters

Plot each region in its physical coordinates, colored by cluster, to check the clusters form coherent anatomy.

In [ ]:
samples = adata.obs["sample"].cat.categories if hasattr(adata.obs["sample"], "cat") \
    else sorted(adata.obs["sample"].unique())
n = len(samples)
ncol = min(2, n)
nrow = int(np.ceil(n / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(6 * ncol, 6 * nrow), squeeze=False)

palette = sc.pl.palettes.default_20
clusters = list(adata.obs["leiden"].cat.categories)
color_map = {c: palette[i % len(palette)] for i, c in enumerate(clusters)}

for ax, s in zip(axs.ravel(), samples):
    sub = adata[adata.obs["sample"] == s]
    xy = sub.obsm["spatial"]
    cols = sub.obs["leiden"].map(color_map).values
    ax.scatter(xy[:, 0], xy[:, 1], c=cols, s=2, linewidths=0)
    ax.set_title(s, fontsize=10); ax.set_aspect("equal"); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
for ax in axs.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 3.7 Save

In [ ]:
out_path = out_dir / "sc_all_clustered.h5ad"
adata.write_h5ad(out_path)
print("Wrote", out_path)
print(adata)